# [7.1] Logit Lens, Tuned Lens, and Patchscopes - Solutions

By the end of this notebook, you will have shown that residual-stream activations in a trained transformer become increasingly decodable as the model computes the next token, and that Patchscope-style activation insertion carries answer information beyond text-only prompt priors.

**Core question:** when a decoder says an activation means a token, what evidence makes that claim stronger than a prompt prior or a pretty story?

```yaml
gt_tier: GT-1 activation-to-language preflight
exercise_id: 7_1_logit_lens_tuned_lens_and_patchscopes
expected_runtime: 60-90 minutes for local exercises; several minutes for the CUDA GELU-1L signature result
requires_gpu: true for the TransformerLens signature result
```

## Learning Objectives

- Implement logit lens, tuned lens, attention lens, and Patchscope-style activation insertion from tensors.
- Build toy oracles where every expected token and rank is known before touching a real model.
- Fit a ridge tuned lens on a train split and judge it only on held-out positions.
- Compare Patchscope answers against text-only prompts, counterfactual activations, and random activations.
- Produce a visible table/plot with real `gelu-1l` prompts, tokens, decoded answers, and controls.

<details>
<summary>Expected output</summary>

The local tests should each print an `All tests ... passed!` line. The signature result should show real `gelu-1l` token rows, a bar plot where tuned-lens held-out accuracy beats ordinary logit lens, and a Patchscope table where patched activations beat the same target prompt without the activation.

</details>

<details>
<summary>Help - how to read this notebook</summary>

Treat every decoded token as a hypothesis about what a specific decoder can extract. The claim becomes stronger only when it survives a held-out split, a text-only baseline, a counterfactual activation, and a random-activation confidence check.

</details>


In [ ]:
import logging
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Literal

import matplotlib.pyplot as plt
import torch as t
import torch.nn.functional as F
from IPython.display import display

chapter = "chapter7_activation_to_language"
section = "part1_lenses_patchscopes"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part1_lenses_patchscopes.tests as tests

PatchscopeTemplate = Literal["entity", "next_token", "fact"]
MAIN = __name__ == "__main__"


@dataclass(frozen=True)
class LensAccuracyReport:
    logit_lens_accuracy: float
    tuned_lens_accuracy: float
    improvement: float
    tuned_lens_improves: bool


@dataclass(frozen=True)
class PatchscopeAccuracyReport:
    patchscope_accuracy: float
    text_only_accuracy: float
    improvement: float
    beats_text_only: bool


@dataclass(frozen=True)
class CounterfactualActivationReport:
    original_answer: int
    patched_answer: int
    changed: bool


@dataclass(frozen=True)
class RandomActivationConfidenceReport:
    mean_confidence: float
    max_confidence: float
    passes_low_confidence: bool


### Exercise - implement `logit_lens` and `top_tokens`

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend up to 10 minutes on this exercise.
> ```

The logit lens is the cheapest possible decoder: take a residual vector and project it through the unembedding. This gives a useful baseline because it has no learned calibration layer.

<details>
<summary>Expected output</summary>

The toy residual directions should decode to logits `[[2.0, 0.0, 1.0], [0.0, 3.0, 1.0]]` and top token ids `[[0], [1]]`.

```text
All tests in `test_logit_lens_and_top_tokens_match_reference` passed!
All tests in `test_top_tokens_rejects_invalid_k` passed!
```

</details>

<details>
<summary>Help - what this proves</summary>

A high-ranked token is linearly decodable through the unembedding at this activation. That is weaker than "the model believes this token", because later computation can still change the prediction.

</details>

Common bug: taking `softmax` before the matrix multiply. Rank logits first; probabilities are only for display.

<details>
<summary>Solution</summary>

```python
def logit_lens(residual_stream: t.Tensor, unembedding: t.Tensor) -> t.Tensor:
    if residual_stream.shape[-1] != unembedding.shape[0]:
        raise ValueError("residual_stream last dimension must match unembedding rows.")
    return residual_stream.float() @ unembedding.float()


def top_tokens(logits: t.Tensor, *, k: int = 5) -> tuple[t.Tensor, t.Tensor]:
    if k <= 0 or k > logits.shape[-1]:
        raise ValueError("k must be between 1 and vocab size.")
    probs = F.softmax(logits.float(), dim=-1)
    values, indices = probs.topk(k=k, dim=-1)
    return indices, values
```

</details>


In [ ]:
def logit_lens(residual_stream: t.Tensor, unembedding: t.Tensor) -> t.Tensor:
    if residual_stream.shape[-1] != unembedding.shape[0]:
        raise ValueError("residual_stream last dimension must match unembedding rows.")
    return residual_stream.float() @ unembedding.float()


def top_tokens(logits: t.Tensor, *, k: int = 5) -> tuple[t.Tensor, t.Tensor]:
    if k <= 0 or k > logits.shape[-1]:
        raise ValueError("k must be between 1 and vocab size.")
    probs = F.softmax(logits.float(), dim=-1)
    values, indices = probs.topk(k=k, dim=-1)
    return indices, values


tests.test_logit_lens_and_top_tokens_match_reference(logit_lens, top_tokens)
tests.test_top_tokens_rejects_invalid_k(top_tokens)


### Exercise - build a top-token table with target ranks

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend up to 10 minutes on this exercise.
> ```

ARENA notebooks should make the model's behavior visible. A rank table is more useful than a naked tensor because it shows the decoded strings, target token, and whether the target was rank 1 or merely somewhere in the tail.

<details>
<summary>Expected output</summary>

The controlled rows should show decoded token strings in probability order, with target ranks `1` and `2`.

```text
All tests in `test_top_token_table_reports_target_ranks` passed!
```

</details>

<details>
<summary>Help - target rank matters</summary>

Accuracy hides near misses. A target rank of 2 means the decoder almost recovered the expected token; a target rank of 20 means the interpretation is much weaker even if the target probability looks nonzero.

</details>

<details>
<summary>Solution</summary>

```python
def top_token_table(logits, id_to_token, *, k=5, target_token_ids=None, row_labels=None):
    if logits.ndim != 2:
        raise ValueError("top_token_table expects logits with shape [row, vocab].")
    if target_token_ids is not None and target_token_ids.shape != logits.shape[:1]:
        raise ValueError("target_token_ids must have one id per logits row.")
    if row_labels is not None and len(row_labels) != logits.shape[0]:
        raise ValueError("row_labels must have one label per logits row.")
    top_ids, top_probs = top_tokens(logits, k=k)
    sorted_ids = logits.float().argsort(dim=-1, descending=True)
    def decode(token_id: int) -> str:
        return str(id_to_token(token_id) if callable(id_to_token) else id_to_token[token_id])
    rows = []
    for row in range(logits.shape[0]):
        target_id = None if target_token_ids is None else int(target_token_ids[row].item())
        target_rank = None
        target_token = None
        if target_id is not None:
            target_token = decode(target_id)
            target_rank = int((sorted_ids[row] == target_id).nonzero(as_tuple=False)[0].item()) + 1
        rows.append({
            "row": row if row_labels is None else row_labels[row],
            "top_ids": [int(x) for x in top_ids[row].tolist()],
            "top_tokens": [decode(int(x)) for x in top_ids[row].tolist()],
            "top_probs": [float(x) for x in top_probs[row].tolist()],
            "target_token": target_token,
            "target_rank": target_rank,
        })
    return rows
```

</details>


In [ ]:
def top_token_table(
    logits: t.Tensor,
    id_to_token,
    *,
    k: int = 5,
    target_token_ids: t.Tensor | None = None,
    row_labels: list[str] | None = None,
) -> list[dict[str, object]]:
    if logits.ndim != 2:
        raise ValueError("top_token_table expects logits with shape [row, vocab].")
    if target_token_ids is not None and target_token_ids.shape != logits.shape[:1]:
        raise ValueError("target_token_ids must have one id per logits row.")
    if row_labels is not None and len(row_labels) != logits.shape[0]:
        raise ValueError("row_labels must have one label per logits row.")

    top_ids, top_probs = top_tokens(logits, k=k)

    def decode(token_id: int) -> str:
        if callable(id_to_token):
            return str(id_to_token(token_id))
        return str(id_to_token[token_id])

    rows: list[dict[str, object]] = []
    sorted_ids = logits.float().argsort(dim=-1, descending=True)
    for row in range(logits.shape[0]):
        target_id = None if target_token_ids is None else int(target_token_ids[row].item())
        target_rank = None
        target_token = None
        if target_id is not None:
            target_token = decode(target_id)
            target_rank = int((sorted_ids[row] == target_id).nonzero(as_tuple=False)[0].item()) + 1
        rows.append(
            {
                "row": row if row_labels is None else row_labels[row],
                "top_ids": [int(x) for x in top_ids[row].tolist()],
                "top_tokens": [decode(int(x)) for x in top_ids[row].tolist()],
                "top_probs": [float(x) for x in top_probs[row].tolist()],
                "target_token": target_token,
                "target_rank": target_rank,
            }
        )
    return rows


tests.test_top_token_table_reports_target_ranks(top_token_table)


### Exercise - implement tuned-lens decoding and held-out accuracy

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 15 minutes on this exercise.
> ```

A tuned lens is a learned affine correction before unembedding. Because it learns parameters, the held-out split is part of the claim.

<details>
<summary>Expected output</summary>

The toy correction should make tuned-lens accuracy `1.0` while ordinary logit lens gets `0.0`.

```text
All tests in `test_tuned_lens_improves_over_logit_lens_on_toy_targets` passed!
All tests in `test_tuned_lens_uses_bias_and_leading_dims` passed!
All tests in `test_prediction_accuracy_rejects_shape_mismatch` passed!
```

</details>

<details>
<summary>Help - where the bias goes</summary>

Apply the bias in residual space, before the unembedding. Adding it to vocabulary logits would learn a token prior rather than a residual-stream correction.

</details>

<details>
<summary>Solution</summary>

```python
def tuned_lens(residual_stream, lens_weight, lens_bias, unembedding):
    if residual_stream.shape[-1] != lens_weight.shape[0]:
        raise ValueError("residual_stream last dimension must match lens_weight rows.")
    transformed = residual_stream.float() @ lens_weight.float()
    if lens_bias is not None:
        transformed = transformed + lens_bias.to(transformed.device)
    return logit_lens(transformed, unembedding)


def prediction_accuracy(logits, target_token_ids):
    if logits.shape[:-1] != target_token_ids.shape:
        raise ValueError("target_token_ids must match logits leading dimensions.")
    return logits.argmax(dim=-1).eq(target_token_ids).float().mean().item()


def lens_accuracy_report(logit_lens_logits, tuned_lens_logits, target_token_ids):
    logit_acc = prediction_accuracy(logit_lens_logits, target_token_ids)
    tuned_acc = prediction_accuracy(tuned_lens_logits, target_token_ids)
    return LensAccuracyReport(logit_acc, tuned_acc, tuned_acc - logit_acc, tuned_acc > logit_acc)
```

</details>


In [ ]:
def tuned_lens(
    residual_stream: t.Tensor,
    lens_weight: t.Tensor,
    lens_bias: t.Tensor | None,
    unembedding: t.Tensor,
) -> t.Tensor:
    if residual_stream.shape[-1] != lens_weight.shape[0]:
        raise ValueError("residual_stream last dimension must match lens_weight rows.")
    transformed = residual_stream.float() @ lens_weight.float()
    if lens_bias is not None:
        transformed = transformed + lens_bias.to(transformed.device)
    return logit_lens(transformed, unembedding)


def prediction_accuracy(logits: t.Tensor, target_token_ids: t.Tensor) -> float:
    if logits.shape[:-1] != target_token_ids.shape:
        raise ValueError("target_token_ids must match logits leading dimensions.")
    predictions = logits.argmax(dim=-1)
    return predictions.eq(target_token_ids).float().mean().item()


def lens_accuracy_report(
    logit_lens_logits: t.Tensor,
    tuned_lens_logits: t.Tensor,
    target_token_ids: t.Tensor,
) -> LensAccuracyReport:
    logit_acc = prediction_accuracy(logit_lens_logits, target_token_ids)
    tuned_acc = prediction_accuracy(tuned_lens_logits, target_token_ids)
    improvement = tuned_acc - logit_acc
    return LensAccuracyReport(
        logit_lens_accuracy=logit_acc,
        tuned_lens_accuracy=tuned_acc,
        improvement=improvement,
        tuned_lens_improves=improvement > 0,
    )


tests.test_tuned_lens_improves_over_logit_lens_on_toy_targets(
    logit_lens,
    tuned_lens,
    lens_accuracy_report,
)
tests.test_tuned_lens_uses_bias_and_leading_dims(tuned_lens, prediction_accuracy)
tests.test_prediction_accuracy_rejects_shape_mismatch(prediction_accuracy)


### Exercise - fit a ridge tuned lens, then evaluate held-out points

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 20 minutes on this exercise.
> ```

Now fit the affine correction rather than receiving it by hand. The toy data has an exact ground truth: a swap of two residual directions plus a bias. Your fitted lens should recover that map and solve held-out points that the logit lens misses.

<details>
<summary>Expected output</summary>

The recovered weight should match the swap matrix, the recovered bias should match `[0.5, -0.25]`, and held-out tuned-lens accuracy should be `1.0`.

```text
All tests in `test_fit_ridge_tuned_lens_and_heldout_eval` passed!
```

</details>

<details>
<summary>Help - why ridge here?</summary>

Ridge regression is a stable way to learn an affine map from one activation space to another. The bias column should not be penalized; otherwise the lens can underfit a simple global shift.

</details>

<details>
<summary>Solution</summary>

```python
def fit_ridge_tuned_lens(residual_stream, target_residual_stream, *, ridge=1e-2):
    design = t.cat([residual_stream.float(), t.ones(residual_stream.shape[0], 1, device=residual_stream.device)], dim=1)
    penalty = t.eye(design.shape[1], device=design.device)
    penalty[-1, -1] = 0.0
    solution = t.linalg.solve(design.T @ design + ridge * penalty, design.T @ target_residual_stream.float())
    return solution[:-1], solution[-1]


def evaluate_lens_on_heldout(residual_stream, lens_weight, lens_bias, unembedding, target_token_ids):
    logit_logits = logit_lens(residual_stream, unembedding)
    tuned_logits = tuned_lens(residual_stream, lens_weight, lens_bias, unembedding)
    return lens_accuracy_report(logit_logits, tuned_logits, target_token_ids)
```

</details>


In [ ]:
def fit_ridge_tuned_lens(
    residual_stream: t.Tensor,
    target_residual_stream: t.Tensor,
    *,
    ridge: float = 1e-2,
) -> tuple[t.Tensor, t.Tensor]:
    if residual_stream.ndim != 2 or target_residual_stream.ndim != 2:
        raise ValueError("ridge fitting expects rank-2 [example, d_model] tensors.")
    if residual_stream.shape[0] != target_residual_stream.shape[0]:
        raise ValueError("source and target tensors must have the same number of examples.")
    design = t.cat(
        [
            residual_stream.float(),
            t.ones(residual_stream.shape[0], 1, device=residual_stream.device),
        ],
        dim=1,
    )
    penalty = t.eye(design.shape[1], device=design.device)
    penalty[-1, -1] = 0.0
    solution = t.linalg.solve(
        design.T @ design + ridge * penalty,
        design.T @ target_residual_stream.float(),
    )
    return solution[:-1], solution[-1]


def evaluate_lens_on_heldout(
    residual_stream: t.Tensor,
    lens_weight: t.Tensor,
    lens_bias: t.Tensor | None,
    unembedding: t.Tensor,
    target_token_ids: t.Tensor,
) -> LensAccuracyReport:
    logit_logits = logit_lens(residual_stream, unembedding)
    tuned_logits = tuned_lens(residual_stream, lens_weight, lens_bias, unembedding)
    return lens_accuracy_report(logit_logits, tuned_logits, target_token_ids)


tests.test_fit_ridge_tuned_lens_and_heldout_eval(
    fit_ridge_tuned_lens,
    evaluate_lens_on_heldout,
)


### Exercise - decode attention-weighted values

> ```yaml
> Difficulty: medium
> Importance: medium
>
> You should spend up to 10 minutes on this exercise.
> ```

An attention lens asks what a query read from the value stream. First combine values with the attention pattern; only then decode through the unembedding.

<details>
<summary>Expected output</summary>

The toy attention pattern and values should decode to `[[[1.0, 0.0], [0.25, 1.5]]]`.

```text
All tests in `test_attention_lens_decodes_attention_weighted_values` passed!
All tests in `test_attention_lens_rejects_rank_or_key_mismatch` passed!
```

</details>

<details>
<summary>Help - the important axes</summary>

For each query, the attention distribution weights key positions. The multiplication is `attention_pattern @ value_vectors`, where the key dimension of attention matches the sequence dimension of values.

</details>

<details>
<summary>Solution</summary>

```python
def attention_lens(attention_pattern, value_vectors, unembedding):
    if attention_pattern.ndim != 3 or value_vectors.ndim != 3:
        raise ValueError("attention_pattern and value_vectors must be rank-3 tensors.")
    if attention_pattern.shape[-1] != value_vectors.shape[-2]:
        raise ValueError("attention key dimension must match value sequence length.")
    attended = attention_pattern.float() @ value_vectors.float()
    return logit_lens(attended, unembedding)
```

</details>


In [ ]:
def attention_lens(
    attention_pattern: t.Tensor,
    value_vectors: t.Tensor,
    unembedding: t.Tensor,
) -> t.Tensor:
    if attention_pattern.ndim != 3 or value_vectors.ndim != 3:
        raise ValueError("attention_pattern and value_vectors must be rank-3 tensors.")
    if attention_pattern.shape[-1] != value_vectors.shape[-2]:
        raise ValueError("attention key dimension must match value sequence length.")
    attended = attention_pattern.float() @ value_vectors.float()
    return logit_lens(attended, unembedding)


tests.test_attention_lens_decodes_attention_weighted_values(attention_lens)
tests.test_attention_lens_rejects_rank_or_key_mismatch(attention_lens)


### Exercise - make Patchscope templates and activation replacement explicit

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 15 minutes on this exercise.
> ```

Patchscopes are not just "decode an activation". They insert a source activation into a target prompt. That means the target prompt and the exact patched position are part of the experiment.

<details>
<summary>Expected output</summary>

The templates should preserve `<ACT>`, the helper should replace only the final target-prompt position, and Patchscope accuracy should beat the text-only baseline on the controlled logits.

```text
All tests in `test_patchscope_templates_and_accuracy_report` passed!
All tests in `test_replace_final_position_activation_rejects_bad_shapes` passed!
```

</details>

<details>
<summary>Help - baseline discipline</summary>

If you change the prompt between patched and unpatched runs, you no longer know whether the activation did the work. The baseline should be the same target prompt with no inserted source activation.

</details>

<details>
<summary>Solution</summary>

```python
def patchscope_prompt(template, placeholder="<ACT>"):
    if template == "entity":
        return f"What entity is represented by {placeholder}?"
    if template == "next_token":
        return f"What token will {placeholder} become next?"
    if template == "fact":
        return f"What fact is stored in {placeholder}?"
    raise ValueError("unknown Patchscope template.")


def replace_final_position_activation(activations, source_activation):
    patched = activations.clone()
    patched[0, -1] = source_activation.to(device=activations.device, dtype=activations.dtype)
    return patched
```

</details>


In [ ]:
def patchscope_prompt(template: PatchscopeTemplate, placeholder: str = "<ACT>") -> str:
    if template == "entity":
        return f"What entity is represented by {placeholder}?"
    if template == "next_token":
        return f"What token will {placeholder} become next?"
    if template == "fact":
        return f"What fact is stored in {placeholder}?"
    raise ValueError("unknown Patchscope template.")


def patchscope_accuracy_report(
    patchscope_logits: t.Tensor,
    text_only_logits: t.Tensor,
    target_answer_ids: t.Tensor,
) -> PatchscopeAccuracyReport:
    patchscope_acc = prediction_accuracy(patchscope_logits, target_answer_ids)
    text_only_acc = prediction_accuracy(text_only_logits, target_answer_ids)
    improvement = patchscope_acc - text_only_acc
    return PatchscopeAccuracyReport(
        patchscope_accuracy=patchscope_acc,
        text_only_accuracy=text_only_acc,
        improvement=improvement,
        beats_text_only=improvement > 0,
    )


def replace_final_position_activation(
    activations: t.Tensor,
    source_activation: t.Tensor,
) -> t.Tensor:
    if activations.ndim != 3:
        raise ValueError("activations must have shape [batch, seq, d_model].")
    if source_activation.ndim != 1:
        raise ValueError("source_activation must have shape [d_model].")
    if activations.shape[0] != 1:
        raise ValueError("this teaching helper expects batch size 1.")
    if activations.shape[-1] != source_activation.shape[0]:
        raise ValueError("source_activation dimension must match activations d_model.")
    patched = activations.clone()
    patched[0, -1] = source_activation.to(device=activations.device, dtype=activations.dtype)
    return patched


tests.test_patchscope_templates_and_accuracy_report(
    patchscope_prompt,
    patchscope_accuracy_report,
    replace_final_position_activation,
)
tests.test_replace_final_position_activation_rejects_bad_shapes(
    replace_final_position_activation,
)


### Exercise - bundle counterfactual and random controls

> ```yaml
> Difficulty: medium
> Importance: high
>
> You should spend up to 15 minutes on this exercise.
> ```

The controls are what stop Patchscope outputs from becoming decorative text. You need one report for whether a counterfactual activation changes the answer, one report for whether random activations stay low-confidence, and one bundle that compares patched, text-only, and random paths.

<details>
<summary>Expected output</summary>

Counterfactual logits should change answer `0 -> 1`; uniform random logits over four tokens should have max confidence `0.25`; the bundled Patchscope control should pass.

```text
All tests in `test_counterfactual_and_random_activation_controls` passed!
All tests in `test_patchscope_eval_bundles_controls` passed!
```

</details>

<details>
<summary>Help - what falsifies the story?</summary>

If text-only accuracy is already high, the target prompt is doing the work. If random activations are high-confidence, the decoder or prompt has a strong prior. If a counterfactual activation does not change the answer, the decoder may not be sensitive to the activation.

</details>

<details>
<summary>Solution</summary>

```python
def counterfactual_activation_report(original_logits, patched_logits):
    original_answer = int(original_logits.argmax().item())
    patched_answer = int(patched_logits.argmax().item())
    return CounterfactualActivationReport(original_answer, patched_answer, original_answer != patched_answer)


def random_activation_confidence_report(random_logits, *, max_allowed_confidence=0.6):
    confidence = F.softmax(random_logits.float(), dim=-1).max(dim=-1).values
    return RandomActivationConfidenceReport(confidence.mean().item(), confidence.max().item(), confidence.max().item() <= max_allowed_confidence)


def patchscope_eval(patchscope_logits, text_only_logits, random_logits, target_answer_ids, *, max_allowed_random_confidence=0.6):
    patchscope = patchscope_accuracy_report(patchscope_logits, text_only_logits, target_answer_ids)
    random_control = random_activation_confidence_report(random_logits, max_allowed_confidence=max_allowed_random_confidence)
    return {"patchscope_accuracy": patchscope.patchscope_accuracy, "text_only_accuracy": patchscope.text_only_accuracy, "improvement": patchscope.improvement, "beats_text_only": patchscope.beats_text_only, "random_mean_confidence": random_control.mean_confidence, "random_max_confidence": random_control.max_confidence, "random_passes_low_confidence": random_control.passes_low_confidence, "passes": patchscope.beats_text_only and random_control.passes_low_confidence}
```

</details>


In [ ]:
def counterfactual_activation_report(
    original_logits: t.Tensor,
    patched_logits: t.Tensor,
) -> CounterfactualActivationReport:
    if original_logits.ndim != 1 or patched_logits.ndim != 1:
        raise ValueError("original_logits and patched_logits must be rank-1 tensors.")
    original_answer = int(original_logits.argmax().item())
    patched_answer = int(patched_logits.argmax().item())
    return CounterfactualActivationReport(
        original_answer=original_answer,
        patched_answer=patched_answer,
        changed=original_answer != patched_answer,
    )


def random_activation_confidence_report(
    random_logits: t.Tensor,
    *,
    max_allowed_confidence: float = 0.6,
) -> RandomActivationConfidenceReport:
    confidence = F.softmax(random_logits.float(), dim=-1).max(dim=-1).values
    mean_confidence = confidence.mean().item()
    max_confidence = confidence.max().item()
    return RandomActivationConfidenceReport(
        mean_confidence=mean_confidence,
        max_confidence=max_confidence,
        passes_low_confidence=max_confidence <= max_allowed_confidence,
    )


def patchscope_eval(
    patchscope_logits: t.Tensor,
    text_only_logits: t.Tensor,
    random_logits: t.Tensor,
    target_answer_ids: t.Tensor,
    *,
    max_allowed_random_confidence: float = 0.6,
) -> dict[str, object]:
    patchscope = patchscope_accuracy_report(
        patchscope_logits,
        text_only_logits,
        target_answer_ids,
    )
    random_control = random_activation_confidence_report(
        random_logits,
        max_allowed_confidence=max_allowed_random_confidence,
    )
    return {
        "patchscope_accuracy": patchscope.patchscope_accuracy,
        "text_only_accuracy": patchscope.text_only_accuracy,
        "improvement": patchscope.improvement,
        "beats_text_only": patchscope.beats_text_only,
        "random_mean_confidence": random_control.mean_confidence,
        "random_max_confidence": random_control.max_confidence,
        "random_passes_low_confidence": random_control.passes_low_confidence,
        "passes": patchscope.beats_text_only and random_control.passes_low_confidence,
    }


tests.test_counterfactual_and_random_activation_controls(
    counterfactual_activation_report,
    random_activation_confidence_report,
)
tests.test_patchscope_eval_bundles_controls(patchscope_eval)


### Exercise - whole-notebook smoke contract

> ```yaml
> Difficulty: easy
> Importance: high
>
> You should spend up to 10 minutes on this exercise after the previous tests pass.
> ```

This cell composes the functions you wrote into the same local contract used by the Python solution file. It is intentionally small: if this fails, debug the earlier sub-function tests first.

<details>
<summary>Expected output</summary>

```text
All tests in `test_notebook_contract` passed!
```

</details>

<details>
<summary>Help - why one more test?</summary>

Sub-function tests catch local bugs; the notebook contract catches integration mistakes, such as forgetting to include the table result or using a different threshold in the control bundle.

</details>


In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    residual = t.tensor([[1.0, 0.0], [0.0, 1.0]])
    unembedding = t.tensor([[2.0, 0.0, 1.0], [0.0, 3.0, 1.0]])
    logits = logit_lens(residual, unembedding)
    top_ids, top_probs = top_tokens(logits, k=1)
    top_table = top_token_table(
        logits,
        [" floor", " sky", " neutral"],
        k=2,
        target_token_ids=t.tensor([0, 1]),
        row_labels=["residual direction 0", "residual direction 1"],
    )

    toy_residual = t.tensor([[1.0, 0.0], [0.0, 1.0]])
    toy_unembedding = t.eye(2)
    logit_logits = logit_lens(toy_residual, toy_unembedding)
    tuned_logits = tuned_lens(
        toy_residual,
        t.tensor([[0.0, 1.0], [1.0, 0.0]]),
        None,
        toy_unembedding,
    )
    targets = t.tensor([1, 0])

    train = t.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])
    true_weight = t.tensor([[0.0, 1.0], [1.0, 0.0]])
    true_bias = t.tensor([0.5, -0.25])
    target_residuals = train @ true_weight + true_bias
    weight, bias = fit_ridge_tuned_lens(train, target_residuals, ridge=0.0)
    heldout_report = evaluate_lens_on_heldout(
        t.tensor([[2.0, 0.0], [0.0, 2.0]]),
        weight,
        bias,
        t.eye(2),
        t.tensor([1, 0]),
    )

    patchscope_logits = t.tensor([[2.0, 0.0], [0.0, 2.0]])
    text_only_logits = t.tensor([[0.0, 2.0], [0.0, 2.0]])
    patch_targets = t.tensor([0, 1])
    patched_acts = replace_final_position_activation(
        t.zeros(1, 3, 2),
        t.tensor([1.0, -1.0]),
    )
    patch_eval = patchscope_eval(
        patchscope_logits,
        text_only_logits,
        t.zeros(3, 4),
        patch_targets,
        max_allowed_random_confidence=0.3,
    )

    return {
        "logit_lens": {
            "logits": logits.tolist(),
            "top_ids": top_ids.tolist(),
            "top_probs": top_probs.tolist(),
            "top_table": top_table,
        },
        "tuned_lens": lens_accuracy_report(
            logit_logits,
            tuned_logits,
            targets,
        ).__dict__,
        "fit_ridge_tuned_lens": heldout_report.__dict__,
        "attention_lens": {
            "logits": attention_lens(
                t.tensor([[[1.0, 0.0], [0.0, 1.0]]]),
                t.tensor([[[1.0, 0.0], [0.0, 1.0]]]),
                t.eye(2),
            ).tolist(),
        },
        "patchscope": {
            "entity_prompt": patchscope_prompt("entity"),
            "next_token_prompt": patchscope_prompt("next_token"),
            "fact_prompt": patchscope_prompt("fact"),
            "patched_final_activation": patched_acts[0, -1].tolist(),
            **patchscope_accuracy_report(
                patchscope_logits,
                text_only_logits,
                patch_targets,
            ).__dict__,
        },
        "patchscope_eval": patch_eval,
        "counterfactual": counterfactual_activation_report(
            t.tensor([2.0, 0.0]),
            t.tensor([0.0, 3.0]),
        ).__dict__,
        "random_confidence": random_activation_confidence_report(
            t.zeros(3, 4),
            max_allowed_confidence=0.3,
        ).__dict__,
    }


tests.test_notebook_contract(run_smoke_test)


## Signature Result - real `gelu-1l` lens and Patchscope controls

The toy tests gave you ground truth. Now run the same mechanics on a pinned public TransformerLens checkpoint. This is the main visible result of the notebook, not a verification-report wrapper.

The claim is deliberately narrow: on this small model and prompt set, a ridge tuned lens improves held-out agreement with the model's own final-token predictions, and Patchscope activation insertion beats the same text-only target prompt.

<details>
<summary>Expected output</summary>

You should see:

- A top-token table with real prompt labels and decoded token strings.
- A lens bar plot where tuned lens beats ordinary logit lens on held-out positions.
- A Patchscope table where patched activations recover the source-prompt target and text-only prompts do not.
- A controls plot showing low random-activation confidence and a counterfactual decoded-token change.

</details>

<details>
<summary>Interpreting the result</summary>

The tuned lens is not magic; it learns a calibration from early residual states to final normalized residual states. The Patchscope result is strongest only because it is compared to the same target prompt without the source activation and because random activations stay low-confidence.

</details>


In [ ]:
TL_GELU1L_MODEL_NAME = "gelu-1l"
TL_GELU1L_HF_ID = "NeelNanda/GELU_1L512W_C4_Code"
TL_GELU1L_REVISION = "bddc0e332f0ae84279e6a6a45d91b314899e1603"
TL_GELU1L_TOKENIZER_ID = "NeelNanda/gpt-neox-tokenizer-digits"
TL_GELU1L_TOKENIZER_REVISION = "0f6671571a20be9756b9991d978047c03b75e749"
TL_BNB_CUDA_OVERRIDE = "130"
TL_RESID_PRE_HOOK = "blocks.0.hook_resid_pre"
TL_RESID_POST_HOOK = "blocks.0.hook_resid_post"
TL_ATTN_PATTERN_HOOK = "blocks.0.attn.hook_pattern"
TL_ATTN_VALUE_HOOK = "blocks.0.attn.hook_v"
TL_LENS_TRAIN_PROMPTS = [
    "The cat sat on the",
    "The bird flew over the",
    "To make tea, boil the",
    "To make bread, bake the",
    "The recipe calls for sugar and",
    "The recipe calls for salt and",
    "The chef cooked a",
    "The teacher taught a",
    "The river flows into the",
    "The road leads into the",
    "The Python function returns a",
    "The HTML page contains a",
    "The file was saved to disk as a",
    "The message was sent by mail as a",
    "The train arrived at the",
    "The plane landed at the",
]
TL_LENS_HELDOUT_PROMPTS = [
    "The programmer wrote a",
    "The singer sang a",
    "The team won the game by scoring a",
    "The team lost the game after missing a",
    "A king lives in a",
    "A student learns in a",
]
TL_PATCHSCOPE_PAIRS = [
    ("The cat sat on the", "The bird flew over the"),
    ("To make tea, boil the", "To make bread, bake the"),
    ("The recipe calls for sugar and", "The recipe calls for salt and"),
    ("The chef cooked a", "The teacher taught a"),
    ("The programmer wrote a", "The singer sang a"),
    ("The team won the game by scoring a", "The team lost the game after missing a"),
]


def load_gelu1l_model_on_cuda():
    if not t.cuda.is_available():
        raise RuntimeError("CUDA is required for the real GELU-1L signature result.")
    os.environ.setdefault("BNB_CUDA_VERSION", TL_BNB_CUDA_OVERRIDE)
    logging.getLogger("bitsandbytes.cextension").setLevel(logging.ERROR)
    from transformer_lens import HookedTransformer

    return HookedTransformer.from_pretrained(
        TL_GELU1L_MODEL_NAME,
        device="cuda",
        dtype="float32",
        tokenizer_name=TL_GELU1L_TOKENIZER_ID,
        tokenizer_revision=TL_GELU1L_TOKENIZER_REVISION,
        revision=TL_GELU1L_REVISION,
    )


def decode_residual(model, residual_stream: t.Tensor) -> t.Tensor:
    normalized = model.ln_final(residual_stream)
    return logit_lens(normalized, model.W_U) + model.b_U


def cache_lens_dataset(model, prompts: list[str]) -> dict[str, object]:
    pre_activations = []
    post_normalized = []
    targets = []
    final_logits = []
    labels = []
    for prompt in prompts:
        tokens = model.to_tokens(prompt)
        with t.inference_mode():
            logits, cache = model.run_with_cache(
                tokens,
                names_filter=lambda name: name in {TL_RESID_PRE_HOOK, TL_RESID_POST_HOOK},
            )
        pre_activations.append(cache[TL_RESID_PRE_HOOK][0].detach())
        post_normalized.append(model.ln_final(cache[TL_RESID_POST_HOOK][0]).detach())
        targets.append(logits[0].argmax(dim=-1).detach())
        final_logits.append(logits[0].detach())
        for pos in range(tokens.shape[1]):
            labels.append(f"{prompt!r} / pos {pos}")
    return {
        "pre": t.cat(pre_activations, dim=0),
        "post_normalized": t.cat(post_normalized, dim=0),
        "targets": t.cat(targets, dim=0),
        "final_logits": t.cat(final_logits, dim=0),
        "labels": labels,
    }


def patchscope_activation_decode(
    model,
    source_activation: t.Tensor,
    target_tokens: t.Tensor,
    *,
    hook_name: str = TL_RESID_POST_HOOK,
) -> t.Tensor:
    def patch_hook(activations: t.Tensor, hook=None) -> t.Tensor:
        _ = hook
        return replace_final_position_activation(activations, source_activation)

    with t.inference_mode():
        patched_logits = model.run_with_hooks(
            target_tokens,
            fwd_hooks=[(hook_name, patch_hook)],
        )
    return patched_logits[0, -1].detach()


def prepare_gelu1l_state() -> dict[str, object]:
    t.cuda.reset_peak_memory_stats()
    model = load_gelu1l_model_on_cuda()
    model.eval()
    train = cache_lens_dataset(model, TL_LENS_TRAIN_PROMPTS)
    heldout = cache_lens_dataset(model, TL_LENS_HELDOUT_PROMPTS)
    lens_weight, lens_bias = fit_ridge_tuned_lens(
        train["pre"],
        train["post_normalized"],
    )
    return {
        "model": model,
        "train": train,
        "heldout": heldout,
        "lens_weight": lens_weight,
        "lens_bias": lens_bias,
    }


def run_gelu1l_signature_result(state: dict[str, object], *, table_rows: int = 8) -> dict[str, object]:
    model = state["model"]
    heldout = state["heldout"]
    lens_weight = state["lens_weight"]
    lens_bias = state["lens_bias"]

    logit_lens_logits = decode_residual(model, heldout["pre"])
    tuned_lens_logits = tuned_lens(
        heldout["pre"],
        lens_weight,
        lens_bias,
        model.W_U,
    ) + model.b_U
    final_decode_logits = logit_lens(heldout["post_normalized"], model.W_U) + model.b_U

    lens_report = lens_accuracy_report(
        logit_lens_logits,
        tuned_lens_logits,
        heldout["targets"],
    )
    final_decode_accuracy = prediction_accuracy(final_decode_logits, heldout["targets"])
    final_decode_error = (final_decode_logits.float() - heldout["final_logits"].float()).abs().max().item()

    top_rows = top_token_table(
        tuned_lens_logits[:table_rows],
        lambda token_id: model.to_string(token_id),
        k=5,
        target_token_ids=heldout["targets"][:table_rows],
        row_labels=heldout["labels"][:table_rows],
    )

    qualitative_rows = []
    for idx in range(min(3, heldout["targets"].numel())):
        qualitative_rows.append(
            {
                "heldout row": heldout["labels"][idx],
                "logit lens top": model.to_string(int(logit_lens_logits[idx].argmax().item())),
                "tuned lens top": model.to_string(int(tuned_lens_logits[idx].argmax().item())),
                "final target": model.to_string(int(heldout["targets"][idx].item())),
                "tuned target rank": top_rows[idx]["target_rank"],
            }
        )

    patchscope_logits = []
    text_only_logits = []
    patchscope_targets = []
    patchscope_rows = []
    counterfactual_report = None
    for clean_prompt, corrupt_prompt in TL_PATCHSCOPE_PAIRS:
        with t.inference_mode():
            clean_logits, clean_cache = model.run_with_cache(
                model.to_tokens(clean_prompt),
                names_filter=lambda name: name == TL_RESID_POST_HOOK,
            )
            corrupt_logits, corrupt_cache = model.run_with_cache(
                model.to_tokens(corrupt_prompt),
                names_filter=lambda name: name == TL_RESID_POST_HOOK,
            )
            target_tokens = model.to_tokens("This means")
            baseline_logits = model(target_tokens)[0, -1].detach()
        target_id = int(clean_logits[0, -1].argmax().item())
        source_activation = clean_cache[TL_RESID_POST_HOOK][0, -1].detach().clone()
        patched_logits = patchscope_activation_decode(model, source_activation, target_tokens)
        patchscope_logits.append(patched_logits)
        text_only_logits.append(baseline_logits)
        patchscope_targets.append(target_id)
        patchscope_rows.append(
            {
                "source prompt": clean_prompt,
                "target token": model.to_string(target_id),
                "patched top token": model.to_string(int(patched_logits.argmax().item())),
                "text-only top token": model.to_string(int(baseline_logits.argmax().item())),
                "patched correct": int(patched_logits.argmax().item()) == target_id,
                "text-only correct": int(baseline_logits.argmax().item()) == target_id,
            }
        )
        if counterfactual_report is None:
            counterfactual_report = counterfactual_activation_report(
                decode_residual(model, clean_cache[TL_RESID_POST_HOOK][0, -1]),
                decode_residual(model, corrupt_cache[TL_RESID_POST_HOOK][0, -1]),
            )

    t.manual_seed(12345)
    random_residuals = (
        t.randn(8, model.cfg.d_model, device=heldout["pre"].device)
        * heldout["pre"].float().std()
        * 0.001
    )
    random_logits = decode_residual(model, random_residuals)
    target_ids = t.tensor(patchscope_targets, device=heldout["pre"].device)
    patchscope_summary = patchscope_eval(
        t.stack(patchscope_logits, dim=0),
        t.stack(text_only_logits, dim=0),
        random_logits,
        target_ids,
        max_allowed_random_confidence=0.2,
    )

    t.cuda.synchronize()
    return {
        "device": t.cuda.get_device_name(0),
        "model": f"{TL_GELU1L_MODEL_NAME} / {TL_GELU1L_HF_ID}",
        "train_prompts": len(TL_LENS_TRAIN_PROMPTS),
        "heldout_prompts": len(TL_LENS_HELDOUT_PROMPTS),
        "heldout_positions": int(heldout["targets"].numel()),
        "lens_report": lens_report.__dict__,
        "final_decode_accuracy": final_decode_accuracy,
        "final_decode_max_abs_error": final_decode_error,
        "top_token_rows": top_rows,
        "qualitative_rows": qualitative_rows,
        "patchscope_rows": patchscope_rows,
        "patchscope_summary": patchscope_summary,
        "counterfactual_original_token": model.to_string(counterfactual_report.original_answer),
        "counterfactual_patched_token": model.to_string(counterfactual_report.patched_answer),
        "peak_vram_gb": t.cuda.max_memory_allocated() / 1024**3,
    }


def run_gpu_test(max_vram_gb: float = 24.0) -> dict[str, object]:
    state = prepare_gelu1l_state()
    result = run_gelu1l_signature_result(state)
    if result["peak_vram_gb"] > max_vram_gb:
        raise AssertionError(f"peak VRAM {result['peak_vram_gb']:.3f} GB exceeded budget {max_vram_gb:.3f} GB")
    return result


def run_full_experiment(max_vram_gb: float = 24.0) -> dict[str, object]:
    return run_gpu_test(max_vram_gb=max_vram_gb)


def decode_play_prompt(state: dict[str, object], prompt: str, *, k: int = 5) -> list[dict[str, object]]:
    model = state["model"]
    tokens = model.to_tokens(prompt)
    with t.inference_mode():
        logits, cache = model.run_with_cache(
            tokens,
            names_filter=lambda name: name == TL_RESID_PRE_HOOK,
        )
    tuned_logits = tuned_lens(
        cache[TL_RESID_PRE_HOOK][0].detach(),
        state["lens_weight"],
        state["lens_bias"],
        model.W_U,
    ) + model.b_U
    labels = [f"{prompt!r} / pos {pos}" for pos in range(tokens.shape[1])]
    return top_token_table(
        tuned_logits,
        lambda token_id: model.to_string(token_id),
        k=k,
        target_token_ids=logits[0].argmax(dim=-1).detach(),
        row_labels=labels,
    )


In [ ]:
gelu1l_state = prepare_gelu1l_state()
signature = run_gelu1l_signature_result(gelu1l_state)

print(f"device: {signature['device']}")
print(f"model: {signature['model']}")
print(f"train prompts: {signature['train_prompts']} | held-out prompts: {signature['heldout_prompts']} | held-out positions: {signature['heldout_positions']}")
print(f"peak VRAM GB: {signature['peak_vram_gb']:.3f}")

display(signature["qualitative_rows"])
display(signature["top_token_rows"])
display(signature["patchscope_rows"])
display(signature["patchscope_summary"])

assert signature["heldout_positions"] >= 20
assert signature["lens_report"]["tuned_lens_improves"]
assert signature["patchscope_summary"]["beats_text_only"]
assert signature["patchscope_summary"]["random_passes_low_confidence"]


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))

lens_names = ["logit lens\npre", "tuned lens\npre", "final decode\npost"]
lens_values = [
    signature["lens_report"]["logit_lens_accuracy"],
    signature["lens_report"]["tuned_lens_accuracy"],
    signature["final_decode_accuracy"],
]
axes[0].bar(lens_names, lens_values, color=["#64748b", "#16a34a", "#2563eb"])
axes[0].set_ylim(0, 1)
axes[0].set_title("Held-out target-token agreement")
axes[0].set_ylabel("top-1 agreement")
for i, value in enumerate(lens_values):
    axes[0].text(i, value + 0.03, f"{value:.3f}", ha="center")

patch_names = ["text-only", "patched"]
patch_values = [
    signature["patchscope_summary"]["text_only_accuracy"],
    signature["patchscope_summary"]["patchscope_accuracy"],
]
axes[1].bar(patch_names, patch_values, color=["#94a3b8", "#7c3aed"])
axes[1].set_ylim(0, 1)
axes[1].set_title("Patchscope target recovery")
for i, value in enumerate(patch_values):
    axes[1].text(i, value + 0.03, f"{value:.3f}", ha="center")

control_names = ["random\nmax conf", "decode\nmax err", "VRAM\nGB"]
control_values = [
    signature["patchscope_summary"]["random_max_confidence"],
    min(signature["final_decode_max_abs_error"] / 1e-4, 1.0),
    signature["peak_vram_gb"],
]
axes[2].bar(control_names, control_values, color=["#f97316", "#0f766e", "#334155"])
axes[2].axhline(0.2, color="#f97316", linewidth=1, linestyle="--", label="random conf threshold")
axes[2].set_title("Controls and resource use")
axes[2].legend(loc="upper right", fontsize=8)

fig.tight_layout()
plt.show()


## Try It Yourself - change the prompt, then interpret the ranks

Change `PLAY_PROMPT` to a prompt you care about, rerun the cell, and inspect whether the tuned lens top token matches the model's final target token. Good prompts here are short next-token contexts; long prompts are fine, but the table gets longer.

<details>
<summary>Interpretation checklist</summary>

- Does the target token reach rank 1, or is it only nearby?
- Are the top alternatives semantically related or just frequent tokens?
- Does changing one word in the prompt change the top decoded tokens?
- Would a text-only prompt or random activation explain the same answer?

</details>


In [ ]:
PLAY_PROMPT = "A student learns in a"  # Change this.
PLAY_TOP_K = 5

play_rows = decode_play_prompt(gelu1l_state, PLAY_PROMPT, k=PLAY_TOP_K)
display(play_rows)


## Limitations

This notebook validates mechanics, not a broad benchmark. The real result uses one small public TransformerLens checkpoint, a small generated prompt set, and targets defined by the model's own final-logit argmax rather than human labels. A tuned lens can improve held-out decoding without proving that decoded words are the model's belief. Patchscope success here is scoped to one hook, one neutral target prompt, and six controlled source prompts; broader claims need more templates, layers, model families, and adversarial controls.
